# 03 · Reconcile saved probability calibration

Audits the earlier SHARP/AIA/fusion calibration package against the frozen master table; no refitting.

**Checkpoint date:** 2026-10-05.

In [ ]:
#!/usr/bin/env python3
"""
Read-only audit of the saved Notebook-18 probability-calibration package
against the frozen 72-hour Gray-Box master prediction table.

No fitting, no GPU, no deletion, no GCS mutation.
"""

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

CAL = Path("/home/abmoses2000/multimodal72_calibration_v1_20261005T082136654618Z")
MASTER_DIR = Path("/home/abmoses2000/graybox_aia72_master_predictions_v2_20261005")
MASTER = MASTER_DIR / "graybox_aia72_master_predictions.csv.gz"
MASTER_MANIFEST = MASTER_DIR / "manifest.json"

OUT = Path("/home/abmoses2000/graybox_aia72_calibration_audit_20261005")
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing audit directory: {OUT}")
OUT.mkdir(parents=True)

EXPECTED = {
    "summary.json": "1670",  # size is informational only
    "selection.json": None,
    "source_receipt.json": None,
    "inner_selection_scores.csv": None,
    "config.json": None,
    "calibration_cases.csv.gz": None,
}

KNOWN_OUTPUT_SHA = {
    "source_receipt.json": "6648f9516bbc9c3448e5789e771eefc86f6478d390d2f6e53035b9e7055c0e87",
    "inner_selection_scores.csv": "7771c6ce428b6a155ab4da035cad8b2afa4d06a3e41503ee8741389046995fa2",
    "config.json": "000bce843d1c60644d27107441e7cdebad0649412b82cd8b71f29b795b7af916",
    "selection.json": "8570bfdfdc6583490d85f4a67c3ebcf73458a59ffa25cd9e57ecf455ca5aaedd",
    "calibration_cases.csv.gz": "3f8ed1512ba915dedf6536cf10390f6c772a958bc94409a203d5edc166203f13",
}
EXPECTED_MASTER_SHA = "17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46"

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def require(cond, msg):
    if not cond:
        raise ValueError(msg)

print("===== FILE INTEGRITY =====")
for name in EXPECTED:
    p = CAL / name
    require(p.is_file(), f"Missing calibration file: {p}")
    actual = sha256(p)
    known = KNOWN_OUTPUT_SHA.get(name)
    if known:
        require(actual == known, f"SHA mismatch for {name}")
    print(name, p.stat().st_size, actual)

require(MASTER.is_file(), f"Missing master table: {MASTER}")
master_sha = sha256(MASTER)
require(master_sha == EXPECTED_MASTER_SHA, "Master prediction table SHA changed")
print("master sha256:", master_sha)

summary = json.loads((CAL / "summary.json").read_text())
selection = json.loads((CAL / "selection.json").read_text())
source_receipt = json.loads((CAL / "source_receipt.json").read_text())
config = json.loads((CAL / "config.json").read_text())
scores = pd.read_csv(CAL / "inner_selection_scores.csv")
cal = pd.read_csv(CAL / "calibration_cases.csv.gz", low_memory=False)
master = pd.read_csv(MASTER, low_memory=False)

print("\n===== SAVED CALIBRATION STATUS =====")
print("status:", summary.get("status"))
print("requested_cases:", summary.get("requested_cases"))
print("input_failure_cases:", summary.get("input_failure_cases"))
print("model_refits:", summary.get("model_refits"))
print("GPU_calls:", summary.get("GPU_calls"))
print("evaluation_roles_read:", summary.get("evaluation_roles_read"))
print("selected_methods:", summary.get("selected_methods"))

require(summary.get("status") == "completed_earlier_matched_calibration_not_evaluation",
        "Unexpected calibration status")
require(summary.get("requested_cases") == 2831, "Requested calibration count changed")
require(summary.get("input_failure_cases") == 0, "Input failures changed")
require(summary.get("model_refits") == 0, "Unexpected model refit")
require(summary.get("GPU_calls") == 0, "Unexpected GPU calls")
require(summary.get("evaluation_roles_read") == [], "Later evaluation roles were read")
require(summary.get("selected_methods") == {"sharp":"raw","aia":"raw","sharp_aia":"raw"},
        "Selected methods changed")
require(selection.get("selected_methods") == summary.get("selected_methods"),
        "Summary and selection disagree")

print("\n===== MASTER ROLE MATCH =====")
mcal = master.loc[master["role"].astype(str) == "probability_calibration"].copy()
print("master probability_calibration rows:", len(mcal))
print("saved calibration rows:", len(cal))

require(len(mcal) == 2831, "Master probability_calibration row count changed")
require(len(cal) == 2831, "Saved calibration_cases row count changed")
require(cal["forecast_case_id"].is_unique, "Saved calibration case IDs not unique")
require(mcal["forecast_case_id"].is_unique, "Master calibration case IDs not unique")

master_ids = mcal["forecast_case_id"].astype(str).tolist()
cal_ids = cal["forecast_case_id"].astype(str).tolist()
require(cal_ids == master_ids, "Saved calibration case IDs/order do not exactly match frozen master role")
print("case IDs/order exact match: True")

print("\n===== LABEL CONSISTENCY =====")
if "label" in mcal.columns and "label" in cal.columns:
    a = pd.to_numeric(mcal["label"], errors="coerce").to_numpy()
    b = pd.to_numeric(cal["label"], errors="coerce").to_numpy()
    require(np.array_equal(a, b, equal_nan=True), "Calibration labels disagree with master")
    print("labels exact match: True")
else:
    print("label comparison unavailable; columns:", list(mcal.columns), list(cal.columns))

print("\n===== AIA RAW CONSISTENCY =====")
master_prob_candidates = [c for c in ["probability", "aia_raw", "probability_mean"] if c in mcal.columns]
require(master_prob_candidates, f"No recognizable AIA probability column in master: {list(mcal.columns)}")
master_prob_col = master_prob_candidates[0]
require("aia_raw" in cal.columns, "Saved calibration_cases lacks aia_raw")

p_master = pd.to_numeric(mcal[master_prob_col], errors="coerce").to_numpy(float)
p_cal = pd.to_numeric(cal["aia_raw"], errors="coerce").to_numpy(float)
require(np.array_equal(p_master, p_cal, equal_nan=True) or np.allclose(p_master, p_cal, equal_nan=True, atol=0, rtol=1e-12),
        f"Saved aia_raw does not match master {master_prob_col}")
print(f"aia_raw matches master column '{master_prob_col}': True")

print("\n===== FIXED FUSION CHECK =====")
require(all(c in cal.columns for c in ["sharp_raw","aia_raw","sharp_aia_raw"]),
        "Missing branch columns in calibration_cases")
expected_fusion = 0.5 * pd.to_numeric(cal["sharp_raw"], errors="coerce").to_numpy(float) + \
                  0.5 * pd.to_numeric(cal["aia_raw"], errors="coerce").to_numpy(float)
actual_fusion = pd.to_numeric(cal["sharp_aia_raw"], errors="coerce").to_numpy(float)
require(np.array_equal(expected_fusion, actual_fusion, equal_nan=True) or
        np.allclose(expected_fusion, actual_fusion, equal_nan=True, atol=1e-15, rtol=1e-12),
        "Saved fusion is not the fixed 0.5/0.5 probability mean")
print("fixed 0.5/0.5 fusion exact/close match: True")

print("\n===== METHOD-SELECTION REPLAY =====")
rows = []
for branch in ["sharp","aia","sharp_aia"]:
    g = scores.loc[scores["branch"] == branch].copy()
    require(set(g["method"]) == {"raw","platt","isotonic"}, f"Candidate methods changed for {branch}")
    g["_rank"] = g["method"].map({"raw":0, "platt":1, "isotonic":2})
    chosen = g.sort_values(["brier","log_loss","_rank"]).iloc[0]["method"]
    saved = selection["selected_methods"][branch]
    require(chosen == saved, f"Selection replay mismatch for {branch}: {chosen} != {saved}")
    best = g.sort_values(["brier","log_loss","_rank"]).iloc[0]
    rows.append({
        "branch": branch,
        "selected_method": saved,
        "selection_cases": int(best["cases"]),
        "selection_positive": int(best["positive"]),
        "brier": float(best["brier"]),
        "log_loss": float(best["log_loss"]),
        "ece_equal_width_10": float(best["ece_equal_width_10"]),
    })
    print(branch, "->", saved, "brier", float(best["brier"]), "log_loss", float(best["log_loss"]))

print("\n===== FINAL-FIT SUPPORT =====")
support = selection["matched_support"]
print(json.dumps(support, indent=2))
require(support["final_fit"]["cases"] == 2831, "Final-fit count changed")
require(support["final_fit"]["positive"] == 184, "Final-fit positive count changed")
require(selection["requested_cases"] == 2831, "Selection requested count changed")
require(selection["input_failure_cases"] == 0, "Selection input failures changed")
require(selection["evaluation_roles_read"] == [], "Selection read later roles")
require(selection["fusion_weights"] == {"sharp":0.5,"aia":0.5}, "Fusion weights changed")

print("\n===== SOURCE RECEIPT =====")
print(json.dumps(source_receipt, indent=2))

report = {
    "status": "CALIBRATION_PACKAGE_RECONCILED_WITH_FROZEN_MASTER",
    "scientific_acceptance": False,
    "master_sha256": master_sha,
    "calibration_package": str(CAL),
    "calibration_case_rows": 2831,
    "calibration_case_ids_exact_master_match": True,
    "labels_match_master": True if ("label" in mcal.columns and "label" in cal.columns) else None,
    "aia_raw_matches_master": True,
    "master_aia_probability_column": master_prob_col,
    "fixed_fusion_0p5_0p5_verified": True,
    "selected_methods": selection["selected_methods"],
    "selection_replay": rows,
    "final_fit_support": support,
    "model_refits": 0,
    "GPU_calls": 0,
    "evaluation_roles_read": [],
    "note": "Audit only. Frozen saved calibrators are reusable; no refitting performed."
}

(OUT / "audit.json").write_text(json.dumps(report, indent=2) + "\n")
pd.DataFrame(rows).to_csv(OUT / "selection_replay.csv", index=False)

print("\n===== AUDIT RESULT =====")
print(json.dumps(report, indent=2))
print("\nSaved:", OUT / "audit.json")
print("Saved:", OUT / "selection_replay.csv")
